# ADENLY — ЛЗ №6: нақты Backlog және шартты Монте-Карло
Бағалау күні: **09.10.2026**. 10 GitHub User Story, жалпы үлгілік баға **53 SP**.

Тіркеу (#1) және JWT кіруі (#2) API қабылдау критерийлері бойынша тексерілген. Қалған 8 User Story толық қабылданбаған. Олардың кодының бір бөлігі жұмыс істейді. Ашық Issue саны әзірлеу жұмысының толық көлемін өлшемейді.

Дәлел: [PR #11](https://github.com/Arnurzhairbek/Adenly/pull/11), [CI](https://github.com/Arnurzhairbek/Adenly/actions). PostgreSQL-пен 28 API тексеруі. SMTP жеткізуі мен браузер UI толық тексерілмеген.

**Тарихи апталық throughput жоқ. Төмендегі P85/P95 — тек жорамалдық сценарий.** Бұрынғы 50/21/29 тапсырма мен [2,3,4,3,5,4] нақты статистика ретінде қолданылмайды.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

TOTAL_WEEKS = 14  # оқу жоспары
CURRENT_WEEK = 6  # оқу тапсырмасындағы жорамал, күннен есептелмейді
REMAINING_WEEKS = TOTAL_WEEKS - CURRENT_WEEK
AUDIT_DATE = "2026-10-09"
rows = [
 (1, "Тіркеу", 3, "Closed"),
 (2, "Жүйеге кіру және JWT", 3, "Closed"),
 (3, "Қорғалған маршруттар мен құқықтар", 5, "Open"),
 (4, "Пайдаланушы профилі", 3, "Open"),
 (5, "Тіл курстарының тізімі", 5, "Open"),
 (6, "Курс сабақтарын қарау", 5, "Open"),
 (7, "Сабақ мазмұны мен материалдар", 8, "Open"),
 (8, "Тапсырмалар мен тесттер", 8, "Open"),
 (9, "Оқу прогресі және жетістіктер", 8, "Open"),
 (10, "Хабарландырулар", 5, "Open")
]
backlog = pd.DataFrame(rows, columns=["Issue", "User Story", "SP үлгісі", "Мәртебе"])
TOTAL_TASKS = len(backlog)
COMPLETED_TASKS = int((backlog["Мәртебе"] == "Closed").sum())
BACKLOG = TOTAL_TASKS - COMPLETED_TASKS
total_sp = int(backlog["SP үлгісі"].sum())
assert (TOTAL_TASKS, COMPLETED_TASKS, BACKLOG, total_sp) == (10, 2, 8, 53)
display(backlog)
print(f"Барлығы: {TOTAL_TASKS} User Story. Қабылданған: {COMPLETED_TASKS}. Ашық: {BACKLOG}.")
print("Қалған үлгілік баға:", int(backlog.loc[backlog["Мәртебе"] == "Open", "SP үлгісі"].sum()), "SP")
print("SP командамен әлі расталмаған. User Story саны SP санына тең емес.")

## Нақты тарих пен сценарий кірісі
Нақты жабылған Issues-тің күндері инспекция жасалған күнді көрсетеді. Оларды жобаның алдыңғы алты аптасында орындаған жұмыс жылдамдығы деп санауға болмайды.

Жорамал сценарий: бір аптада 0, 1 немесе 2 **User Story** қабылдануы мүмкін. Әр мән төмендегі тізімнен бірдей ықтималдықпен таңдалады. Нөлдік апта тәуекелін модельге қосамыз. Ірі User Story-лердің көлемі әртүрлі болғандықтан, бұл қарапайым модель нақты уәде бермейді.

In [ ]:
OBSERVED_WEEKLY_THROUGHPUT = None  # нақты деректер жоқ
SCENARIO_THROUGHPUT = [0, 1, 1, 2, 1, 2]  # автордың жорамалы, тарихи дерек емес
history = (SCENARIO_THROUGHPUT if OBSERVED_WEEKLY_THROUGHPUT is None
           else OBSERVED_WEEKLY_THROUGHPUT)
MODEL_TYPE = "ШАРТТЫ СЦЕНАРИЙ" if OBSERVED_WEEKLY_THROUGHPUT is None else "Тарихи деректермен болжам"
assert len(history) > 0 and all(isinstance(x, int) and x >= 0 for x in history)
if not any(x > 0 for x in history) and BACKLOG > 0:
    raise ValueError("Барлық throughput нөл: аяқталу мерзімін есептеу мүмкін емес")
display(pd.DataFrame({"Сценарий нұсқасы": range(1, len(history)+1), "User Story/апта": history}))
print(MODEL_TYPE)
print("Жорамал орташа throughput:", round(float(np.mean(history)), 2), "User Story/апта")

In [ ]:
RUNS = 10000
rng = np.random.default_rng(42)
weeks = np.zeros(RUNS, dtype=int)
remaining = np.full(RUNS, BACKLOG, dtype=int)
while np.any(remaining > 0):
    active = remaining > 0
    remaining[active] -= rng.choice(history, size=int(active.sum()))
    weeks[active] += 1
    if weeks.max() > 1000:
        raise RuntimeError("Симуляция 1000 аптадан асты. Кірістерді тексеріңіз")
quantiles = {p: int(np.quantile(weeks, p/100, method="higher")) for p in [50,70,85,95]}
p85, p95 = quantiles[85], quantiles[95]
probability = float(np.mean(weeks <= REMAINING_WEEKS) * 100)
display(pd.DataFrame({"Көрсеткіш": [f"P{p}" for p in quantiles], "Қажет апта": list(quantiles.values())}))
print(f"{MODEL_TYPE}: {REMAINING_WEEKS} аптада аяқталу ықтималдығы {probability:.2f}%")
print("P85: сценарийлердің кемінде 85%-ы осы аптаға дейін аяқталады.")
print("Мерзім жорамал throughput-қа тәуелді. Нақты міндеттеме емес.")

In [ ]:
unique_weeks = np.arange(weeks.min(), weeks.max()+1)
probabilities = np.array([np.mean(weeks <= w) * 100 for w in unique_weeks])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(weeks, bins=np.arange(weeks.min(), weeks.max()+2)-0.5, edgecolor="black")
axes[0].set(xlabel="Апта", ylabel="Симуляция саны", title="Шартты сценарийдің мерзімдері")
axes[1].plot(unique_weeks, probabilities, marker="o")
axes[1].axhline(85, linestyle=":", label="85%")
axes[1].axvline(REMAINING_WEEKS, linestyle="--", label=f"{REMAINING_WEEKS} апта")
axes[1].set(xlabel="Апта", ylabel="Аяқталу ықтималдығы (%)", ylim=(0,105), title="Шартты жинақталған ықтималдық")
axes[1].legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame({"Апта":unique_weeks, "Шартты ықтималдық (%)":np.round(probabilities,2)}))

## Planning Poker хаттамасы
Команданың нақты дауыс беруі туралы дерек берілмеген. Төмендегі кесте — толтыруға дайын хаттама. 53 SP бұрынғы оқу үлгісінен сақталды. Қатысушылардың дауыстарын немесе келісімін ойдан қоспаймыз.

Талқылауға арналған сұрақтар: email жеткізуі, құқық матрицасы, профильдің сақталуы, мобильді тексеру, API жолдарының сәйкестігі, материал қауіпсіздігі, қайта тапсыру ережесі, прогресс пен жетістік шарттары.

In [ ]:
poker = backlog[["Issue", "User Story", "SP үлгісі"]].copy()
for name in ["Арнұр", "Шыңғыс", "Абдулазиз", "Қайсар"]:
    poker[name] = pd.NA
poker["Келіспеушілік себебі"] = "Команда талқылауында толтырылады"
poker["Қорытынды SP"] = pd.NA
poker["Шешім мәртебесі"] = "Команда растауы қажет"
display(poker)
print("53 SP — оқу үлгісі. Нақты команда консенсусы әлі жоқ.")

In [ ]:
summary = pd.DataFrame({
 "Көрсеткіш": ["Жалпы User Story", "Қабылданған", "Ашық", "Үлгілік SP", "Қалған оқу аптасы", "Болжам түрі", "P85", "P95", "8 аптада аяқталу"],
 "Нәтиже": [str(TOTAL_TASKS), str(COMPLETED_TASKS), str(BACKLOG), str(total_sp), str(REMAINING_WEEKS), MODEL_TYPE, f"{p85} апта", f"{p95} апта", f"{probability:.2f}%"]
})
display(summary)
print("Ұсыныс: алдымен профильді нақты сақтау мен қалған қабылдау тексерулерін аяқтау.")
print("Апта сайын acceptance criteria бойынша жабылған Issues санын тіркеу.")
print("Нақты throughput жиналғанда сценарий кірістерін ауыстырып, болжамды қайта есептеу.")